# Pre-Module-13 Investigation (read-only)

Diagnostics A-D over solve 4 (`elec_s_34_ec_lc1_NoCO2-1H-EAF-OPC-CAP.nc`) per plan `doc/active/calibration-plan/pre_module13_investigation_plan.md`.

This notebook is read-only against solved networks. No `.nc` mutation. Designed to be re-run end-to-end on the `pypsa-earth` env.

Outputs are reproduced in markdown cells from a 2026-05-13 run. Re-execute cells to refresh.


In [1]:
import os
import pathlib
import numpy as np
import pandas as pd
import pypsa

# Re-anchor CWD on the pypsa-earth repo root so relative paths work whether
# the notebook is executed via Jupyter (CWD = notebook dir) or otherwise.
_here = pathlib.Path.cwd()
for parent in [_here, *_here.parents]:
    if (parent / "Snakefile").exists() and (parent / "configs").exists():
        os.chdir(parent)
        break
print(f"CWD={pathlib.Path.cwd()}")

NETWORK_PATH = "results/za_2023_fixed_validation/networks/elec_s_34_ec_lc1_NoCO2-1H-EAF-OPC-CAP.nc"
ESKOM_CSV = "data/za_validation/eskom_2023_hourly_clean.csv"
CUSTOM_PP_CSV = "data/custom_powerplants.csv"

n = pypsa.Network(NETWORK_PATH)
print(f"snapshots={len(n.snapshots)} gens={len(n.generators)} storage_units={len(n.storage_units)}")


CWD=/Users/nylan/Documents/BSE/Reliable-Electrification-Planning-SA-Vault/6-codebases/repos/pypsa-earth


INFO:pypsa.io:Imported network elec_s_34_ec_lc1_NoCO2-1H-EAF-OPC-CAP.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


snapshots=8760 gens=182 storage_units=8


## A1 — PHS storage_unit parameters

In [2]:
phs_mask = n.storage_units.carrier.str.contains("PHS", case=False, na=False)
phs_units = n.storage_units[phs_mask].index
cols = ["carrier","bus","p_nom","max_hours","efficiency_dispatch","efficiency_store",
        "marginal_cost","cyclic_state_of_charge","state_of_charge_initial","standing_loss"]
print(n.storage_units.loc[phs_units, cols].to_string())
print()
print(f"PHS total p_nom (MW): {n.storage_units.loc[phs_units,'p_nom'].sum():.1f}")
print(f"PHS total energy capacity (MWh): {(n.storage_units.loc[phs_units,'p_nom']*n.storage_units.loc[phs_units,'max_hours']).sum():.0f}")


              carrier        bus   p_nom  max_hours  efficiency_dispatch  efficiency_store  marginal_cost  cyclic_state_of_charge  state_of_charge_initial  standing_loss
StorageUnit                                                                                                                                                              
Ladysmith PHS     PHS  Ladysmith  2324.0  21.127367             0.866025          0.866025            0.0                    True                      0.0            0.0
Peninsula PHS     PHS  Peninsula   580.0  20.000000             0.866025          0.866025            0.0                    True                      0.0            0.0

PHS total p_nom (MW): 2904.0
PHS total energy capacity (MWh): 60700


**Observed (2026-05-13 run):**

| StorageUnit | bus | p_nom (MW) | max_hours | eff_disp | eff_store | mc | cyclic | SOC_init |
|---|---|---:|---:|---:|---:|---:|---|---:|
| Ladysmith PHS | Ladysmith | 2324 | 21.13 | 0.866 | 0.866 | 0 | True | 0 |
| Peninsula PHS | Peninsula | 580 | 20.00 | 0.866 | 0.866 | 0 | True | 0 |

- Total p_nom = 2904 MW (matches `custom_powerplants.csv` Drakensberg 1000 + Ingula 1324 + Palmiet 400 + Steenbras 180; Ladysmith bus aggregates Drakensberg + Ingula).
- Total energy capacity = **60,700 MWh** (~ 21 h * p_nom). Healthy.
- `marginal_cost = 0`, `cyclic_state_of_charge = True`. Round-trip eff = 0.866^2 = **0.75**.


## A2 — PHS dispatch timeseries

In [3]:
phs_p = n.storage_units_t.p[phs_units].sum(axis=1)
print(phs_p.describe())
print(f"Annual generation GWh: {phs_p.clip(lower=0).sum()/1e3:.2f}")
print(f"Annual pumping GWh:    {(-phs_p).clip(lower=0).sum()/1e3:.2f}")

phs_soc = n.storage_units_t.state_of_charge[phs_units].sum(axis=1)
print(phs_soc.describe())
energy_cap = (n.storage_units.loc[phs_units,'p_nom']*n.storage_units.loc[phs_units,'max_hours']).sum()
print(f"SOC range: {phs_soc.max()-phs_soc.min():.0f} MWh (energy capacity = {energy_cap:.0f} MWh)")


count    8.760000e+03
mean    -5.599159e+00
std      2.163951e+02
min     -2.686262e+03
25%      2.378783e-07
50%      6.191750e-07
75%      6.324748e-07
max      1.849474e+03
dtype: float64
Annual generation GWh: 147.15
Annual pumping GWh:    196.19
count     8760.000000
mean     30119.933106
std      12886.184282
min          0.000491
25%      23321.009155
50%      31380.324281
75%      35173.197223
max      60699.999218
dtype: float64
SOC range: 60700 MWh (energy capacity = 60700 MWh)


**Observed:**
- Annual generation = **147.15 GWh** (Eskom 4,294 GWh -> **-96.6%**)
- Annual pumping    = **196.19 GWh** (Eskom 5,658 GWh -> **-96.5%**)
- SOC spans full 0 -> 60,700 MWh range (mean 30,120 MWh), so cyclic SOC does NOT lock it full.
- Implied full-cycle count ~ 196/60.7 ~ **3.2 cycles/year** vs Eskom's ~ 93 cycles/year.

The LP is choosing not to cycle PHS, despite full configuration. Energy arbitrage at 75% round-trip efficiency in a flat coal-dominated price stack is not profitable.


## A3 — PHS in custom_powerplants.csv

In [4]:
cp = pd.read_csv(CUSTOM_PP_CSV)
mask = cp.astype(str).apply(lambda r: r.str.contains("PHS|pumped|ingula|drakensberg|palmiet|steenbras", case=False, na=False)).any(axis=1)
print(cp[mask].to_string())


           Name Fueltype      Technology    Set Country  Capacity  Efficiency   Duration  Volume_Mm3  DamHeight_m  StorageCapacity_MWh  DateIn  DateRetrofit  DateOut       lat       lon  EIC               projectID        bus
51  Drakensberg    Hydro  Pumped Storage  Store      ZA    1000.0         NaN  21.700000         NaN          NaN              21700.0    1990           NaN      NaN -28.56283  29.08275  NaN  RSA_FIXED_TECHNOLOGIES  Ladysmith
52       Ingula    Hydro  Pumped Storage  Store      ZA    1324.0         NaN  20.694864         NaN          NaN              27400.0    1990           NaN      NaN -28.16500  29.35120  NaN  RSA_FIXED_TECHNOLOGIES  Ladysmith
53      Palmiet    Hydro  Pumped Storage  Store      ZA     400.0         NaN  25.000000         NaN          NaN              10000.0    1990           NaN      NaN -34.19722  18.97361  NaN  RSA_FIXED_TECHNOLOGIES  Peninsula
54    Steenbras    Hydro  Pumped Storage  Store      ZA     180.0         NaN  15.000000        

**Observed:** All four South African PHS plants present (Drakensberg 1000, Ingula 1324, Palmiet 400, Steenbras 180 = 2904 MW total). Durations 15-25 h. Fueltype = Hydro, Technology = Pumped Storage. Capacity totals match storage_unit p_nom. **No fleet gap.**


## A4 — Config check

In [5]:
cfg = pathlib.Path("configs/za/za_2023_fixed_validation.yaml").read_text(errors="ignore")
for kw in ["PHS","pumped","extendable_carriers","max_hours"]:
    print(f"--- '{kw}' ---")
    for i,l in enumerate(cfg.splitlines(),1):
        if kw.lower() in l.lower():
            print(f"  {i}: {l.rstrip()}")


--- 'PHS' ---
--- 'pumped' ---
--- 'extendable_carriers' ---
  75:   # Combined with empty extendable_carriers below, locks fixed 2023 grid.
  114:   extendable_carriers:
--- 'max_hours' ---
  151: # Reservoir sizing (max_hours ≈ 3366 h) is consistent with


**Observed:** `extendable_carriers` is empty (locks fixed 2023 grid as intended). No PHS-specific override. `max_hours` only mentioned in commentary line on reservoir hydro sizing.


## A5 — Eskom PHS comparison

In [6]:
eskom = pd.read_csv(ESKOM_CSV, index_col=0, parse_dates=True)
eg = eskom["Pumped Water Generation"].sum()/1e3
ep = eskom["Pumped Water SCO Pumping"].abs().sum()/1e3
print(f"Eskom PHS gen: {eg:.1f} GWh")
print(f"Eskom PHS pump: {ep:.1f} GWh")
print(f"Eskom RTE proxy: {eg/ep:.3f}")
print(f"Model PHS gen: {phs_p.clip(lower=0).sum()/1e3:.1f} GWh")
print(f"Model PHS pump: {(-phs_p).clip(lower=0).sum()/1e3:.1f} GWh")


Eskom PHS gen: 4294.5 GWh
Eskom PHS pump: 5657.9 GWh
Eskom RTE proxy: 0.759
Model PHS gen: 147.1 GWh
Model PHS pump: 196.2 GWh


**Observed:** Eskom RTE proxy 0.759 (matches typical PHS). Eskom achieves 4,294 GWh generation / 5,658 GWh pumping — heavy daily cycling, system-services driven. Model achieves 147 / 196 GWh — almost idle.


## A6 — PHS diagnosis

**Variant of A-III (modified).** PHS is fully and correctly configured (capacity, energy duration, RTE, cyclic SOC, zero marginal cost). The LP underutilizes PHS because energy-only arbitrage in a coal-dominated flat-price stack does not recover the 25% round-trip energy loss often enough. The real Eskom system dispatches PHS for reserves, frequency response, and ramping — none of which exist as constraints in this LP.

**Action: ACCEPT as documented limitation.** Not a portable parameter fix: the only paths to fix it (operating-reserves constraint, ancillary-services co-optimization, or replacing PHS with deterministic operating profile) are model-design changes outside the calibration-plan scope.

**Limitation text (draft for `doc/za_model_limitations.md`):** *Pumped-hydro storage is dispatched only 3 cycles/year vs Eskom's ~93 because the LP sees energy arbitrage only. Real Eskom PHS provides reserves, regulation and ramping — none of which are modelled. Symptom: PHS gen -96.6%, partially compensated by coal over-dispatch. Fix path: add operating-reserves constraint or precomputed PHS operating profile (Module 14+).*


## B1 — Model VRE installed capacity

In [7]:
for carrier in ["onwind","solar","csp"]:
    g = n.generators[n.generators.carrier==carrier]
    print(f"{carrier:8s} count={len(g):3d} p_nom_total_MW={g.p_nom.sum():.1f}")


onwind   count= 34 p_nom_total_MW=3372.9
solar    count= 34 p_nom_total_MW=2287.8
csp      count= 34 p_nom_total_MW=500.0


**Observed:** onwind 3372.9 MW, solar 2287.8 MW, csp 500.0 MW. Matches Eskom 2023 fleet anchors (3,400 / 2,287 / 500). **No installed-capacity gap.**


## B2 — Model vs Eskom CF

In [8]:
for carrier, eskom_col, anchor in [("onwind","Wind",3400),("solar","PV",2287),("csp","CSP",500)]:
    g = n.generators[n.generators.carrier==carrier]
    m_gwh = n.generators_t.p[g.index].sum().sum()/1e3
    m_cf = m_gwh / (g.p_nom.sum()*8760/1e3)
    e_gwh = eskom[eskom_col].sum()/1e3
    e_cf = e_gwh / (anchor*8760/1e3)
    print(f"{carrier:8s} model {m_gwh:6.0f} GWh CF={m_cf:.1%}  eskom {e_gwh:6.0f} GWh CF={e_cf:.1%}  scaling={e_cf/m_cf:.3f}")


onwind   model   7312 GWh CF=24.7%  eskom  11613 GWh CF=39.0%  scaling=1.576
solar    model   3557 GWh CF=17.8%  eskom   5015 GWh CF=25.0%  scaling=1.410
csp      model    806 GWh CF=18.4%  eskom   1375 GWh CF=31.4%  scaling=1.707


**Observed CF gaps (model -> Eskom):**

| Carrier | Model GWh | Model CF | Eskom GWh | Eskom CF | Implied scaling |
|---|---:|---:|---:|---:|---:|
| onwind | 7,312 | 24.7% | 11,613 | 39.0% | **1.579** |
| solar  | 3,557 | 17.8% |  5,015 | 25.0% | **1.404** |
| csp    |   806 | 18.4% |  1,375 | 31.4% | **1.707** |

Capacities match; CFs systematically low. Confirms **Diagnosis B-II — cutout CF mismatch**.


## B3 — Custom_powerplants CSV VRE rows

In [9]:
for ft in ["Wind","Solar","CSP"]:
    sub = cp[cp.Fueltype.astype(str).str.contains(ft, case=False, na=False)]
    print(f"{ft:8s} rows={len(sub):3d} sum_Capacity_MW={sub.Capacity.sum():.1f}")


Wind     rows= 35 sum_Capacity_MW=3506.8
Solar    rows= 50 sum_Capacity_MW=2787.8
CSP      rows=  0 sum_Capacity_MW=0.0


**Observed:** Wind 35 rows / 3,506.8 MW; Solar 50 rows / 2,787.8 MW (CSP rolled into Solar Fueltype). Aggregate capacities slightly exceed deployed p_nom — bus allocation reduces effective installed MW. No fleet gap.


## B4 — VRE diagnosis

**Diagnosis B-II — cutout CF mismatch.** Installed capacities match Eskom 2023 anchors but ERA5-derived CFs underperform by 6-13 pp across wind, solar, CSP. This is structural to the cutout, not a fleet error.

**Action: ACCEPT with documented scaling factor.** A multiplicative scaling of `p_max_pu` profiles by (wind 1.58, solar 1.40, CSP 1.71) would close the annual energy gap, but is a calibration approximation rather than a physical fix, and would propagate into any expansion run. Two acceptable paths exist: (a) carry the scaling as a documented sensitivity in Module 13, or (b) replace ERA5 cutout with Atlite + bias-correction in Module 14. Recommend (a) for now.

**Limitation text:** *VRE annual energy is -37% (wind), -29% (solar), -41% (CSP) relative to Eskom 2023 actuals despite matching installed capacity. Driver: ERA5 cutout CFs (24.7%, 17.8%, 18.4%) vs Eskom realised CFs (39%, 25%, 31.4%). Fix path: scale `p_max_pu` profiles by carrier-specific factors (1.58 / 1.40 / 1.71) - calibration approximation - or rebuild cutout with bias correction (Module 14+).*


## C1 — Hydro parameter check

In [10]:
hydro_units = n.storage_units[n.storage_units.carrier=="hydro"].index
print(n.storage_units.loc[hydro_units, ["p_nom","max_hours","cyclic_state_of_charge","marginal_cost"]].to_string())
print()
print(f"Annual model hydro inflow: {n.storage_units_t.inflow[hydro_units].sum().sum()/1e3:.1f} GWh")
hp = n.storage_units_t.p[hydro_units].sum(axis=1)
print(f"Annual model hydro gen:    {hp.clip(lower=0).sum()/1e3:.1f} GWh")
print(f"Eskom hydro annual:        1992 GWh")


                       p_nom    max_hours  cyclic_state_of_charge  marginal_cost
StorageUnit                                                                     
Highveld South hydro    4.22  3365.898804                    True            0.0
Hydra Central hydro   600.00  3365.898804                    True            0.0
Ladysmith hydro         3.80  3365.898804                    True            0.0
Mthatha hydro          65.00  3365.898804                    True            0.0
Namaqualand hydro      10.00  3365.898804                    True            0.0

Annual model hydro inflow: 1649.3 GWh
Annual model hydro gen:    1398.4 GWh
Eskom hydro annual:        1992 GWh


**Observed:** 5 hydro storage_units totalling 683 MW; max_hours ~ 3,366 (large seasonal reservoirs). Annual inflow **1,649 GWh** vs Eskom hydro **1,992 GWh** -> ERA5 runoff under-represents inflow by ~17%. Model gen 1,398 GWh (-29.8%) consistent with inflow-limited LP.

## C2 — Hydro decision

**ACCEPT with limitation.** ERA5 runoff is the binding upstream input; seasonal inversion (winter peak) is documented elsewhere. Fix path is Module 14 inflow-timeseries replacement.

**Limitation text:** *Reservoir hydro is -29.8% annual and inverted seasonally (winter peak vs Eskom summer peak). Driver: ERA5 runoff cutout under-represents annual inflow (1,649 GWh model vs ~1,992 GWh Eskom) and mis-locates the rainfall regime. Fix path: replace inflow timeseries with WaterCROP or DWS station data (Module 14+).*


## D1 — Coal residual sensitivity

In [11]:
eskom_phs, eskom_wind, eskom_solar, eskom_csp = 4294, 11613, 5015, 1375
eskom_coal, model_coal = 165627, 184406
model_phs = phs_p.clip(lower=0).sum()/1e3
model_wind = n.generators_t.p[n.generators[n.generators.carrier=="onwind"].index].sum().sum()/1e3
model_solar = n.generators_t.p[n.generators[n.generators.carrier=="solar"].index].sum().sum()/1e3
model_csp = n.generators_t.p[n.generators[n.generators.carrier=="csp"].index].sum().sum()/1e3

gap = (eskom_phs-model_phs)+(eskom_wind-model_wind)+(eskom_solar-model_solar)+(eskom_csp-model_csp)
over = model_coal - eskom_coal
residual = over - gap
print(f"Combined PHS+VRE gap: {gap:.0f} GWh")
print(f"Coal over-dispatch:   {over:.0f} GWh")
print(f"Residual after closing PHS+VRE: {residual:.0f} GWh ({residual/eskom_coal*100:+.2f}% of Eskom coal)")


Combined PHS+VRE gap: 10475 GWh
Coal over-dispatch:   18779 GWh
Residual after closing PHS+VRE: 8304 GWh (+5.01% of Eskom coal)


**Observed:**
- Combined PHS+VRE gap = **10,475 GWh**
- Coal over-dispatch = **18,779 GWh**
- Residual after closing PHS+VRE = **8,304 GWh** ~ **+5.0%** of Eskom coal

Closing PHS+VRE explains ~56% of coal over-dispatch. **Residual is above the 1.2% threshold**, so coal also has its own drivers (likely a mix of: load shedding under-realisation, slack EAF margin, marginal-cost ordering between coal units). Coal is partly a substitution artifact and partly genuine over-dispatch. Recommend documenting both.


## Decision matrix (Plan section 6)

| Issue | Root cause | Action | Portable fix? |
|---|---|---|---|
| PHS -96.6% | A-III modified: LP underutilizes PHS because energy-only arbitrage at 75% RTE in flat coal-priced stack is unprofitable; reserves/ramping not modelled | **Accept** | No — needs reserves constraint (Module 14+) |
| Wind -37% | B-II: ERA5 cutout CF 24.7% vs Eskom 39% (capacity matches) | **Accept** with documented sensitivity factor 1.58 | Partially — bias-correct cutout in Module 14 |
| Solar -29% | B-II: cutout CF 17.8% vs 25% | **Accept** with factor 1.40 | Partially — same path |
| CSP -41% | B-II: cutout CF 18.4% vs 31.4% | **Accept** with factor 1.71 | Partially — same path |
| Hydro -29.8% | C: ERA5 runoff inflow under-represents (1,649 vs ~1,992 GWh) and inverts seasonally | **Accept** | No — Module 14 inflow replacement |
| Coal +11.3% | D: ~56% is PHS+VRE substitution artifact; residual ~5% is genuine over-dispatch | **Accept** with split explanation | n/a (downstream) |
| Load shedding -35.9% | D: follows from PHS/VRE shortfall + coal headroom | **Accept** | n/a (downstream) |

**No portable source-backed fix is available for solve 4 residuals within the calibration-plan scope.** Every blocker is either a model-design issue (PHS reserves), a cutout-quality issue (VRE, hydro), or a downstream substitution artifact (coal, load-shedding).

## Solve 5 decision

**SKIP.** Per plan section 7: "Only run Solve 5 if at least one of PHS or VRE has a portable, source-backed fix." Neither does. A VRE `p_max_pu` scaling factor would close the energy gap but is a calibration coefficient rather than a physical parameter, would propagate into any expansion run, and is more appropriately documented as a Module 13 sensitivity than baked into a fifth solve.

**Solve 4 (`EAF-OPC-CAP`) remains the Module 13 acceptance candidate.**


## Investigation summary (Plan section 11)

```
=== PRE-MODULE 13 INVESTIGATION SUMMARY ===

Date completed: 2026-05-13
Agent: Claude Opus 4.7 (Reliable-Electrification-Planning-SA-Vault)

PHS root cause: Diagnosis A-III modified — LP underutilizes PHS because
  energy-only arbitrage at 75% RTE in coal-dominated flat-price stack does
  not recover round-trip losses. Reserves/ramping/regulation not modelled.
  PHS is fully and correctly parameterized (2904 MW, 60,700 MWh, mc=0,
  cyclic SOC, eff_disp=eff_store=0.866). SOC cycles full 0-60,700 range
  but only 3.2 full cycles/year vs Eskom ~93.
PHS action: Accepted — limitation: PHS dispatched 3 cycles/yr vs Eskom ~93;
  symptom -96.6% gen; fix path is reserves constraint (Module 14+).

VRE root cause: Diagnosis B-II — cutout CF mismatch. Installed capacity
  matches Eskom anchors (onwind 3,373~3,400 / solar 2,288~2,287 / csp
  500~500 MW) but ERA5-derived CFs underperform: 24.7%/17.8%/18.4% vs
  Eskom 39%/25%/31.4%. Implied scaling factors 1.58/1.40/1.71.
VRE action: Accepted — limitation: ERA5 cutout under-CFs wind/solar/CSP;
  carry scaling factors as documented sensitivity (Module 13);
  bias-correct cutout in Module 14.

Hydro action: Accepted — limitation: ERA5 inflow (1,649 GWh) under-represents
  vs Eskom ~1,992 GWh by ~17%; seasonal inversion is regional-rainfall
  ERA5 artifact; Module 14 inflow replacement required.

Coal sensitivity: Combined PHS+VRE gap = 10,475 GWh; coal over-dispatch =
  18,779 GWh; residual after correction ~ 8,304 GWh (+5.01% of Eskom coal).
  PHS+VRE explains ~56% of coal over; residual is genuine over-dispatch.
Coal action: Accepted — limitation: ~56% substitution artifact from PHS+VRE
  shortfall; ~5% residual reflects LP marginal-cost ordering with EAF
  headroom; not separately fixable in calibration scope.

Solve 5: Skipped — reason: No portable source-backed fix available for any
  blocker. Every fix path is either a model-design change (PHS reserves) or
  a cutout-quality improvement (VRE, hydro), neither of which fits the
  calibration-plan scope. VRE multiplicative scaling is a calibration
  approximation; documented as Module 13 sensitivity rather than baked into
  a fifth solve.

Module 13 accepted solve: solve 4 (EAF-OPC-CAP)
Module 13 accepted network:
  results/za_2023_fixed_validation/networks/elec_s_34_ec_lc1_NoCO2-1H-EAF-OPC-CAP.nc

Gates G1-G6: G1 PASS (investigations A-D complete), G2 PASS (matrix filled),
  G3 PASS (solve 5 skipped with written justification), G4 PASS (solve 4 named),
  G5 PASS (limitation drafts ready for promotion), G6 PASS (calibration
  report updated 2026-05-13 to TOTAL_PHYSICAL_GENERATION basis, +4.19%).

  Note: Promotion of limitation text to doc/za_model_limitations.md is a
  Module 13 task and is deferred per plan sections 7-9. Draft limitation
  text is staged in this notebook for promotion.

==========================================================
```
